# Day 4 · 04 · Pivot and Reshape

The same data can be laid out **long** (one row per observation) or **wide** (one column per category). Analysis and storage prefer long; reports and comparisons prefer wide. This notebook moves between the two.

**In this notebook**
- Long and wide layouts
- `pivot_table`: totals, several statistics, missing cells
- Row and column percentages
- `pivot` versus `pivot_table`
- `melt`: wide back to long
- `stack` and `unstack`
- `crosstab` for counts
- Flattening `MultiIndex` columns
- Binning with `pd.cut` before pivoting

**Dataset:** the retail CSV files, joined into one `sales` frame

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../datasets").resolve()

orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(DATA / "order_items.csv")
products = pd.read_csv(DATA / "products.csv")
customers = pd.read_csv(DATA / "customers.csv")

sales = (
    items.merge(orders, on="order_id")
    .merge(products[["product_id", "product_name", "category"]], on="product_id")
    .merge(customers[["customer_id", "country", "segment"]], on="customer_id")
)
sales["revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount"])
valid = sales[sales["status"] != "cancelled"]

## 1. Long and wide
A `groupby` on two keys gives a **long** result: one row per category and channel.

In [2]:
long = valid.groupby(["category", "channel"], as_index=False)["revenue"].sum().round(0)
long.head(6)

,category,channel,revenue
0,Books,mobile,22368.0
1,Books,store,16292.0
2,Books,web,31630.0
3,Electronics,mobile,43582.0
4,Electronics,store,33835.0
5,Electronics,web,68003.0


## 2. `pivot_table`: group and spread in one step
`pivot_table` is a `groupby` whose second key becomes the **columns**.

| Argument | Role |
| --- | --- |
| `index` | row labels |
| `columns` | column labels |
| `values` | the column to aggregate |
| `aggfunc` | how to aggregate (default: mean) |
| `margins=True` | add row and column totals |
| `fill_value` | what to show for empty cells |

In [3]:
table = valid.pivot_table(
    index="category", columns="channel", values="revenue", aggfunc="sum", margins=True, margins_name="Total"
)
table.round(0)

channel,mobile,store,web,Total
category,,,,
Books,22368.0,16292.0,31630.0,70290.0
Electronics,43582.0,33835.0,68003.0,145420.0
Home,18159.0,11194.0,25124.0,54476.0
Sports,21317.0,14184.0,29898.0,65400.0
Toys,12342.0,6802.0,18504.0,37648.0
Total,117768.0,82307.0,173160.0,373235.0


### Percentages of a row or column total
Divide the table by its totals. `div(..., axis=0)` aligns the divisor with the **rows**.

In [4]:
body = valid.pivot_table(index="category", columns="channel", values="revenue", aggfunc="sum")

row_share = body.div(body.sum(axis=1), axis=0).mul(100).round(1)   # each row sums to 100
col_share = body.div(body.sum(axis=0), axis=1).mul(100).round(1)   # each column sums to 100

print("Channel mix within each category (%):")
print(row_share, "\n")
print("Category mix within each channel (%):")
print(col_share)

Channel mix within each category (%):
channel      mobile  store   web
category                        
Books          31.8   23.2  45.0
Electronics    30.0   23.3  46.8
Home           33.3   20.5  46.1
Sports         32.6   21.7  45.7
Toys           32.8   18.1  49.2 

Category mix within each channel (%):
channel      mobile  store   web
category                        
Books          19.0   19.8  18.3
Electronics    37.0   41.1  39.3
Home           15.4   13.6  14.5
Sports         18.1   17.2  17.3
Toys           10.5    8.3  10.7


### Several values and statistics
Pass a dictionary to aggregate each value column in its own way. The columns become a `MultiIndex`: value column on top, channel below.

In [5]:
valid.pivot_table(
    index="category",
    columns="channel",
    values=["revenue", "quantity"],
    aggfunc={"revenue": "sum", "quantity": "sum"},
    fill_value=0,
).round(0)

quantity             revenue                  
channel       mobile store  web   mobile    store      web
category                                                  
Books            633   447  893  22368.0  16292.0  31630.0
Electronics      484   364  819  43582.0  33835.0  68003.0
Home             541   320  783  18159.0  11194.0  25124.0
Sports           466   262  721  21317.0  14184.0  29898.0
Toys             380   196  554  12342.0   6802.0  18504.0

## 3. `pivot` versus `pivot_table`
- `pivot` only **rearranges**. Each index/column pair must occur at most once.
- `pivot_table` **aggregates**, so duplicates are fine.

Use `pivot` when the data is already one row per cell, and you want an error if it is not.

In [6]:
monthly_long = (
    valid.assign(month=valid["order_date"].dt.to_period("M"))
    .groupby(["month", "category"], as_index=False)["revenue"]
    .sum()
)

wide = monthly_long.pivot(index="month", columns="category", values="revenue").round(0)
print(wide.shape)
wide.head(4)

(18, 5)


category,Books,Electronics,Home,Sports,Toys
month,,,,,
2024-01,2061.0,5222.0,1648.0,2331.0,1285.0
2024-02,2720.0,6056.0,1937.0,1344.0,1441.0
2024-03,3190.0,7794.0,2546.0,2716.0,1310.0
2024-04,2354.0,5640.0,2208.0,2409.0,1502.0


In [7]:
# The raw lines have many rows per (date, category), so pivot refuses.
try:
    valid.pivot(index="order_date", columns="category", values="revenue")
except ValueError as error:
    print("ValueError:", error)

ValueError: Index contains duplicate entries, cannot reshape


## 4. `melt`: wide to long
`melt` is the inverse of `pivot`. Column headers become values in a new column.

- `id_vars`: columns to keep as they are
- `var_name`: name for the column that receives the old headers
- `value_name`: name for the column that receives the values

Files from spreadsheets are often wide (one column per month). Melt them before analysis.

In [8]:
back_to_long = wide.reset_index().melt(id_vars="month", var_name="category", value_name="revenue")

print("wide:", wide.shape, "-> long:", back_to_long.shape)
back_to_long.sort_values(["month", "category"]).head(5)

wide: (18, 5) -> long: (90, 3)


,month,category,revenue
0,2024-01,Books,2061.0
18,2024-01,Electronics,5222.0
36,2024-01,Home,1648.0
54,2024-01,Sports,2331.0
72,2024-01,Toys,1285.0


## 5. `stack` and `unstack`
These move a level between the **row index** and the **columns**.
- `unstack` : inner row level → columns (long to wide)
- `stack` : columns → inner row level (wide to long)

In [9]:
grouped = valid.groupby(["segment", "channel"])["revenue"].sum().round(0)
print(grouped.head(4), "\n")

unstacked = grouped.unstack("channel")
print(unstacked, "\n")

print(unstacked.stack().head(4))

segment    channel
Consumer   mobile      67832.0
           store       50557.0
           web        108196.0
Corporate  mobile      35635.0
Name: revenue, dtype: float64 

channel          mobile    store       web
segment                                   
Consumer        67832.0  50557.0  108196.0
Corporate       35635.0  21399.0   48751.0
Small Business  14301.0  10351.0   16212.0 

segment    channel
Consumer   mobile      67832.0
           store       50557.0
           web        108196.0
Corporate  mobile      35635.0
dtype: float64


## 6. `crosstab`: counting combinations
`pd.crosstab(rows, columns)` counts how often each pair occurs. `normalize` turns counts into shares:
- `normalize="index"`: each row sums to 1
- `normalize="columns"`: each column sums to 1

In [10]:
pd.crosstab(orders["channel"], orders["status"], margins=True)

status,cancelled,completed,returned,shipped,All
channel,,,,,
mobile,30,504,36,56,626
store,35,318,20,31,404
web,91,759,56,64,970
All,156,1581,112,151,2000


In [11]:
pd.crosstab(orders["channel"], orders["status"], normalize="index").mul(100).round(1)

status,cancelled,completed,returned,shipped
channel,,,,
mobile,4.8,80.5,5.8,8.9
store,8.7,78.7,5.0,7.7
web,9.4,78.2,5.8,6.6


## 7. Flatten `MultiIndex` columns
Two-level column headers are awkward to export and to select. Join the levels into single names.

In [12]:
stats = valid.pivot_table(index="category", columns="channel", values="revenue", aggfunc=["sum", "mean"]).round(1)
print("before:", stats.columns.tolist()[:3], "...")

stats.columns = [f"{stat}_{channel}" for stat, channel in stats.columns]
stats.reset_index()

before: [('sum', 'mobile'), ('sum', 'store'), ('sum', 'web')] ...


,category,sum_mobile,sum_store,sum_web,mean_mobile,mean_store,mean_web
0,Books,22368.3,16292.0,31629.9,67.8,69.9,67.4
1,Electronics,43582.1,33835.2,68003.1,165.1,191.2,162.3
2,Home,18158.9,11193.6,25123.8,63.3,66.6,58.7
3,Sports,21317.3,14183.8,29898.5,86.7,93.3,82.4
4,Toys,12341.5,6802.1,18504.5,65.6,66.7,67.8


## 8. Bin a number, then pivot
`pd.cut` turns a numeric column into labelled ranges. The bins become a category you can group and pivot by.

- Intervals are closed on the right by default: `(50, 100]`.
- `pd.qcut` makes bins with equal **numbers of rows** instead of equal widths.

In [13]:
order_value = valid.groupby("order_id").agg(value=("revenue", "sum"), channel=("channel", "first"))

order_value["band"] = pd.cut(
    order_value["value"],
    bins=[0, 50, 100, 250, 500, np.inf],
    labels=["up to 50", "50-100", "100-250", "250-500", "over 500"],
)

order_value.pivot_table(index="band", columns="channel", values="value", aggfunc="count", observed=False, margins=True)

channel,mobile,store,web,All
band,,,,
up to 50,99,56,124,279
50-100,127,65,171,363
100-250,213,141,361,715
250-500,105,72,171,348
over 500,52,35,52,139
All,596,369,879,1844


## 9. Ordered labels: weekday by quarter
Text labels sort alphabetically, which puts Friday before Monday. An **ordered categorical** keeps the natural order in every table.

In [14]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

calendar = orders.assign(
    weekday=pd.Categorical(orders["order_date"].dt.day_name(), categories=days, ordered=True),
    quarter=orders["order_date"].dt.to_period("Q").astype("str"),
)

calendar.pivot_table(index="weekday", columns="quarter", values="order_id", aggfunc="count", observed=False)

quarter,2024Q1,2024Q2,2024Q3,2024Q4,2025Q1,2025Q2
weekday,,,,,,
Monday,27,39,37,60,37,42
Tuesday,34,47,38,49,35,58
Wednesday,45,36,33,58,45,45
Thursday,36,41,34,64,55,57
Friday,29,43,54,71,45,70
Saturday,42,56,54,83,62,73
Sunday,28,38,43,52,45,60


## Which tool?
| Goal | Tool |
| --- | --- |
| Summarise long data into a grid | `pivot_table` |
| Rearrange unique rows into a grid | `pivot` |
| Count combinations | `crosstab` |
| Wide columns into rows | `melt` |
| Move an index level to columns, or back | `unstack`, `stack` |
| Numbers into ranges | `cut`, `qcut` |

## Key takeaways
- Keep data long for analysis. Pivot at the end, for presentation.
- `pivot_table` aggregates; `pivot` only reshapes and fails on duplicates.
- `melt` undoes a pivot.
- Flatten two-level column headers before exporting.

## Practice
1. Average discount by category and customer segment, with totals.
2. A table of units sold: products as rows, years as columns.
3. Melt the result of exercise 2 back to long form.
4. Share of orders per status within each country, as percentages.